In [1]:

from pathlib import Path
import duckdb
import polars as pl

# Folder utama proyek
PROJECT_ROOT = Path.cwd().resolve().parent

# Folder yang berisi dataset
DATA_PATH = PROJECT_ROOT / "data" / "raw"

# Cari semua file Parquet
files = sorted(DATA_PATH.glob("*.parquet"))

if not files:
    raise FileNotFoundError(
        f"Tidak ada file Parquet di {DATA_PATH}"
    )

print("Lokasi dataset:", DATA_PATH)
print("Jumlah file:", len(files))

for file in files:
    print(f"{file.name}: {file.stat().st_size / 1_000_000:.1f} MB")


Lokasi dataset: C:\SMESTER 7\ANALISIS BIG DATA\TUGAS 1\tugas1-zeey0809\data\raw
Jumlah file: 2
0000.parquet: 272.4 MB
0001.parquet: 269.0 MB


In [2]:

# Membaca kedua file menggunakan Polars LazyFrame
lazy_data = pl.scan_parquet(
    [str(file) for file in files]
)

# Melihat nama dan tipe kolom
print(lazy_data.collect_schema())

# Menghitung jumlah baris dan nilai kosong
ringkasan = lazy_data.select([
    pl.len().alias("jumlah_baris"),
    pl.all().null_count().name.suffix("_null"),
]).collect()

display(ringkasan)

# Menampilkan 10 baris pertama
contoh_data = lazy_data.head(10).collect()
display(contoh_data)


Schema({'text': String})


jumlah_baris,text_null
u32,u32
5566778,0


text
str
"""Indonesia adalah negara kepula…"
"""Secara geografis, Indonesia be…"
"""Indonesia memiliki iklim tropi…"
"""Sejarah Indonesia sangat panja…"
"""Indonesia memproklamasikan kem…"
"""Peristiwa bersejarah lainnya m…"
"""Sistem pemerintahan Indonesia …"
"""Era Reformasi dimulai tahun 19…"
"""Ekonomi Indonesia terus berkem…"


sys:1: UserWarning: POLARS_MAX_BLOCKING_THREAD_COUNT reached (512), this may indicate a deadlock


In [5]:

connection = duckdb.connect()

# Membaca dan menggabungkan semua file Parquet
file_pattern = str(DATA_PATH / "*.parquet").replace("\\", "/")

profil_duckdb = connection.execute(
    "SUMMARIZE SELECT * FROM read_parquet(?)",
    [file_pattern]
).pl()

display(profil_duckdb)


""
struct[12]
"{""text"",""VARCHAR"","""",""􀀁􀀁􀀁􀀁􀀁􀀁􀀁􀀁􀀁􀀁􀀁􀀁􀀁􀀁􀀁􀀁􀀁􀀁􀀁􀀁􀀁􀀁􀀁􀀁􀀁􀀁􀀁􀀁􀀁􀀁􀀁􀀁􀀁􀀁􀀁􀀁􀀁􀀁􀀁􀀁􀀁􀀁􀀁􀀁􀀁􀀁􀀁􀀁􀀁􀀁􀀁􀀁􀀁􀀁􀀁􀀁􀀁􀀁􀀁􀀁􀀁􀀁􀀁 ML-O155-06 Lantern ML-O155-06 lantern is a newly developed LED Marine Signal Lantern that has 3 intensity outputs ranging 4, 5 and 6 nautical miles for site selection. ML"",4512985,null,null,null,null,null,5566778,0.00}"


### Catatan Temuan Awal — Milestone 1

1. **Ukuran data:** Dataset terdiri dari dua file Parquet dengan total ukuran sekitar 541 MB.
2. **Jumlah baris:** Diisi berdasarkan hasil profiling Polars.
3. **Tipe kolom:** Diisi berdasarkan hasil `collect_schema()`.
4. **Kolom dengan nilai kosong terbanyak:** Diisi berdasarkan hasil perhitungan `null_count`.
5. **Anomali yang perlu ditangani:** Diidentifikasi berdasarkan nilai kosong, data duplikat, tipe data yang tidak sesuai, atau nilai yang tidak wajar apabila ditemukan.